<a href="https://colab.research.google.com/github/marleriee/Teaching-Machine-Learning/blob/main/Lab2_Task7_ExtendedDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 2: Machine Learning in Python: It's easier than it sounds.**

## *Case Description: Extended Dataset*

You are a running coach working within a **large coaching network** that oversees **100,000 competitive female distance runners worldwide**. To better support athlete health and performance, coaches across the network systematically collect both self-reported and observed data into a centralized database.

In our first training group, we focused on only three variables, which gave us a good first understanding of how decision trees work.

**Now, we decided to collect more data in an extended dataset to gain more accurate insights.**


In this lab, you will learn how to get explainable insights using a simple decision tree but also a more advanced machine learning algorithm, based on a combination of many decision trees.


The extended database includes:

*   Training load (running_ten_hours): Whether runners consistently train more than 10 hours per week (Yes/No).

*   Fatigue (fatigue): Self-reported tiredness or lack of recovery (Yes/No).

*   Menstrual disturbance (menstrual_disturbance): For female athletes, presence of irregular or absent periods related to energy availability (Yes/No).

*   Nutrition (calories): Estimated average daily caloric intake.

*   Body weight (body_weight): Self-reported or measured body mass (kg).

*   Shoe technology (carbon_shoe): Use of carbon-plate shoes (Never/Sometimes/Always).

*   Rest strategy (intuitive_rest): Whether the athlete reports resting intuitively (Yes/No).

*   Outcome (overusefracture): Development of overuse fractures (Yes/No).


In [ ]:
# Step 0: If you are new to google colab, you have to run this:
!pip install pandas scikit-learn matplotlib graphviz shap

In [ ]:
# Step 1: Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import graphviz
import shap
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz
import xgboost as xgb

In [ ]:
# Step 2: Load dataset from GitHub
url = "https://raw.githubusercontent.com/marleriee/Teaching-Machine-Learning/main/runners_data_extended.csv"
data = pd.read_csv(url)

# Step 3: Encode categorical variables
# Binary Yes/No variables
binary_cols = ['running_ten_hours', 'fatigue', 'menstrual_disturbance', 'intuitive_rest', 'overusefracture']
for col in binary_cols:
    data[col] = data[col].map({'Yes':1, 'No':0})

# Encode carbon shoes
data = pd.get_dummies(data, columns=['carbon_shoe'], drop_first=True) #instead of having a categorical variable, we now create two variable,
#dropping "never" and creating "carbon_shoe_Sometimes" and "carbon_shoe_Yes", i.e., always

# We can check that the dataset looks similar to the one we used by hand, just including 100,000 runners this time.
print(data.head(10)) # this command shows us the first ten rows of the dataset

In [ ]:
# Step 4: Prepare train/test split

# Separate features (X) and target (y)
X = data.drop("overusefracture", axis=1)   # X = all columns except "overusefracture"
y = data["overusefracture"]                # y = the target column we want to predict
feature_names = list(X.columns)

# Convert boolean columns to integers (0 and 1)
X = X.astype(int)

# Split the dataset into training and testing sets
# - 80% of the data will be used for training (X_train, y_train)
# - 20% will be used for testing (X_test, y_test)
# - random_state=42 ensures reproducibility of the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Task 7.1: Explain what min_samples_split, min_samples_leaf, and max_depth mean in the DecisionTreeClassifier command below. You could annotate the Lab Code using # behind the command.**

*Answer:*

In [ ]:
# Step 5: Train a simple Decision Tree
dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=1000, min_samples_split=10000, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)  # Predicted class labels
y_proba_dt = dt.predict_proba(X_test)[:,1] # Predicted probabilities for the positive class

# Plot the tree
plt.figure(figsize=(30,10))  # Large figure for readability
plot_tree(
    dt,
    feature_names=X_train.columns,  # Show feature names
    class_names=["No Fracture", "Fracture"],  # Label classes
    filled=True,                    # Color nodes by class
    rounded=True,                   # Rounded boxes
    fontsize=12,
    node_ids=True,
)
plt.title("Decision Tree Visualization (max_depth=4)")
plt.show()

In [ ]:
# Print probabilities for each leaf
n_nodes = dt.tree_.node_count
children_left = dt.tree_.children_left
children_right = dt.tree_.children_right
feature = dt.tree_.feature
threshold = dt.tree_.threshold
value = dt.tree_.value

print("\nLeaf node probabilities (Overuse Fracture):")
for i in range(n_nodes):
    if children_left[i] == children_right[i]:  # it's a leaf
        total = value[i].sum()
        prob = value[i][0][1]/total  # probability of fracture
        print(f"Leaf {i}: Probability of fracture = {prob:.2f}")

**Task 7.2: Find the three leafs with the highest fracture risk, and describe runners who are assigned to these leafs. What characterises them? Obs. You need to zoom in into the Figure a lot.**

*Answer:*

In [ ]:
# Step 6: Train an XGBoost Classifier

# XGBoost = "Extreme Gradient Boosting"
# It's a powerful ensemble learning algorithm based on decision trees.
# - Combines many weak learners (trees) to create a strong model.
# - Uses gradient boosting to sequentially improve predictions.
# - Known for high performance on structured/tabular data.

#Think of XGBoost like a team of tiny decision-making trees.
#Each tree looks at the mistakes of the previous ones and tries to correct them. Together, they form a “super-smart” predictor!

xgb_clf = xgb.XGBClassifier(
    use_label_encoder=False,   # Disable older label encoding warning
    eval_metric='auc',         # Metric used during training to monitor performance
    random_state=42            # Ensures reproducible results
)

# Train the model on the training set
xgb_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_xgb = xgb_clf.predict(X_test)            # Predicted class labels
y_proba_xgb = xgb_clf.predict_proba(X_test)[:,1] # Predicted probabilities for the positive class

In [ ]:
# Step 7: Compare performance using common machine learning metrics.
# Ask a LLM if you do not understand the metrics.
def evaluate(y_true, y_pred, y_proba, model_name):
    """
    Evaluate classification performance with metrics relevant for injury prediction.

    Parameters:
    - y_true: True labels (0 = No Fracture, 1 = Fracture)
    - y_pred: Predicted class labels
    - y_proba: Predicted probabilities for the positive class (fracture)
    - model_name: Name of the model (string)

    Explanation of metrics:
    - Accuracy: Overall proportion of correct predictions. Useful as a general measure, but can be misleading if fractures are rare.
    - Precision: Of all runners predicted to get a fracture, how many actually did. High precision → fewer false alarms for athletes.
    - Recall: Of all runners who actually got a fracture, how many did the model correctly identify. High recall → fewer missed injuries.
    - F1 Score: Harmonic mean of precision and recall. Balances false positives and false negatives.
    - ROC AUC: Measures the model's ability to distinguish between fracture vs no fracture across all thresholds. Higher AUC → better discrimination.

    In sports science:
    - Recall is often critical: we want to catch as many at-risk athletes as possible, even if it means some false alarms.
    - Precision matters when intervention is costly or burdensome.
    """

    print(f"\n🔹 Performance for {model_name}:")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred), 3))
    print("Recall:", round(recall_score(y_true, y_pred), 3))
    print("F1 Score:", round(f1_score(y_true, y_pred), 3))
    print("ROC AUC:", round(roc_auc_score(y_true, y_proba), 3))

evaluate(y_test, y_pred_dt, y_proba_dt, "Decision Tree")
evaluate(y_test, y_pred_xgb, y_proba_xgb, "XGBoost")

**Task 7.2: How do the models compare?**


*Answer:*

In [ ]:
# Let's calculate the mean predicted probability by the model across classes.
# Ask a LLM what the code does.

# Create a DataFrame with true labels and predicted probabilities
df = pd.DataFrame({
    "overusefracture": y_test,
    "DT_proba": y_proba_dt,
    "XGB_proba": y_proba_xgb
})

# Group by the observed outcome and calculate mean and SD
summary = df.groupby("overusefracture").agg(
    DT_mean=('DT_proba', 'mean'),
    XGB_mean=('XGB_proba', 'mean'),
)

print(summary)

# Explainable Artificial Intelligence

A single decision tree is very intuitive: it makes predictions by following a sequence of “if-then” rules. You can trace any prediction back to the splits in the tree, see which features mattered, and even visualize the decision path. For example, a tree might say:

“If training load > 50 and menstrual disfunction is present → high risk of fracture.”

You can explain this to anyone—even without a statistics background.

______________

**XGBoost**, on the other hand, is an ensemble method. **It combines hundreds or thousands of small decision trees in a process called gradient boosting.** Each tree only corrects the mistakes of the previous trees. While this makes XGBoost extremely powerful and accurate, it also makes it very opaque:

*   There’s no single “path” to follow for a prediction.

*   The contribution of each feature is spread across many trees.

*   It’s nearly impossible to explain a single prediction just by looking at the model.

This is why XGBoost is considered a black-box model in comparison to a simple decision tree.

________________

##Introducing SHAP Values

To overcome this explainability challenge, we can use **SHAP (SHapley Additive exPlanations)** values.

*   SHAP values come from game theory.

*   They assign each feature a contribution value for a particular prediction, showing how much it increased or decreased the predicted risk.

*   With SHAP, we can open the black box of XGBoost and understand feature importance both globally and locally:

    - Globally: Which features influence predictions most across the dataset.

    - Locally: Why the model predicted a high risk for a specific patient.

*Imagine XGBoost as a team of hundreds of advisors giving a final decision. SHAP values tell you how much each advisor influenced the final decision for each case.*

This way, even though XGBoost is complex, we can make its predictions interpretable and actionable, which is essential in biomedical applications like fracture risk prediction.

In [ ]:
# Subset the test set (e.g., 1000 samples)
subset_size = 1000
X_test_subset = X_test.sample(n=subset_size, random_state=42)

"""
Explanation:
- SHAP values quantify how much each feature contributes to a model's prediction.
- Computing SHAP for the entire dataset (100,000 athletes) can be very slow and memory-intensive.
- Using a subset (1,0000) keeps computations practical while still providing a representative view.
- This approach is common in sports science analytics: we often sample large datasets to get interpretable insights quickly.
"""

# Create a SHAP explainer for the trained XGBoost model
explainer = shap.Explainer(xgb_clf, X_train)

# Compute SHAP values for a subset of the test
shap_values = explainer(X_test_subset)

# Convert SHAP values to a DataFrame for easy handling
shap_df = pd.DataFrame(np.abs(shap_values.values), columns=X_train.columns)

# Compute mean absolute SHAP values across all samples
mean_shap = shap_df.mean().sort_values(ascending=False)

# Bar plot of average absolute SHAP values
plt.figure(figsize=(10,6))
mean_shap.plot(kind='bar', color='skyblue')
plt.ylabel("Mean |SHAP value|")
plt.title("Feature Importance: Average SHAP Values Across the Sample")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Each bar represents a feature from the model.

# Height of the bar = average absolute SHAP value across the subset of samples.

# Interpretation:
# - Taller bars → feature had a stronger influence on model predictions.
# - Shorter bars → feature contributed less to the predictions.
# - Absolute values are used because we care about the magnitude of influence,
#   not the direction (positive or negative).

# Takeaway: This plot helps us understand which features are most important in predicting overuse fractures.

In [ ]:
# Get an insight into how SHAP values are distributed in individuals. Here, we also can see direction.

shap_long = pd.DataFrame(shap_values.values, columns=X_test_subset.columns)
shap_long["sample"] = range(shap_long.shape[0])  # optional: keep sample IDs

shap_melted = shap_long.melt(id_vars="sample", var_name="Feature", value_name="SHAP value")

# Sina plot
plt.figure(figsize=(12,6))
sns.violinplot(x="Feature", y="SHAP value", data=shap_melted, inner=None, color="lightgrey")
sns.stripplot(x="Feature", y="SHAP value", data=shap_melted, size=3, jitter=True, color="blue")
plt.xticks(rotation=45, ha='right')
plt.title(f"Distribution of SHAP Values Across Samples (subset of {subset_size} samples)")
plt.tight_layout()
plt.show()

#The grey violin shows the overall distribution of SHAP values for each feature.
#The blue points are individual sample contributions.
#Taller violins or wider spreads indicate features whose contributions vary a lot across samples.
#Positive SHAP values → feature pushed prediction toward fracture, negative → away from fracture.

#Takeaway: This plot gives both magnitude and variability, helping us understand how each feature influences predictions across the dataset.

In [ ]:
#Let's look at a specific runner to get personalised insights into the fracture risk predictions.

sample_idx = 0  # you can change to any index in the subset
X_sample = X_test_subset.iloc[sample_idx:sample_idx+1]

# Print the feature values
print("Feature values for the sample:\n", X_sample)

# Predicted probability for the positive class (fracture)
predicted_risk = xgb_clf.predict_proba(X_sample)[:,1][0]
print(f"\nPredicted fracture risk for this sample: {predicted_risk:.3f}")


In [ ]:
# Compute SHAP values for that sample
shap_values_sample = explainer(X_sample)
# - This calculates the contribution of each feature to the model’s prediction for this single person.
# - Output is an array of SHAP values, showing how much each feature pushes the prediction up or down.


# Visualize local explanation
shap.initjs()  # enable interactive plots in Jupyter/Colab

shap.force_plot(
    explainer.expected_value,      # base value (average prediction)
    shap_values_sample.values,     # SHAP values for this sample
    X_sample                        # feature values
)

# - The **base value** is where the model would predict without any feature contributions.
# - **Red bars** push the prediction toward higher risk (fracture), **blue bars** push it lower.
# - The **width of each bar** represents the magnitude of that feature’s influence.
# - The final prediction (f(x)) = base value + sum of all SHAP contributions.
# - This plot provides a **local explanation**: it shows why the model predicted this probability for this particular person.

### **Interpreting the personal results in a sports science context**

- The predicted risk (e.g., 0.534) tells us that this runner has an estimated 53% chance of experiencing an overuse fracture.

- The SHAP values show how each feature contributed to increasing or decreasing this risk:
    - Red bars (positive SHAP values) push the prediction toward higher fracture risk.
    - Blue bars (negative SHAP values) push the prediction toward lower risk.
    - Width of each bar indicates the magnitude of the influence.

Practical insights for coaches:

1. Features with high positive SHAP values may represent modifiable risk factors.
   Example:
     - Fatigue = 1 (Yes) → increases fracture risk.
     - Low calorie intake → increases fracture risk.

2. Features with negative SHAP values indicate protective factors:
   Example:
     - Menstrual disturbance = 0 (No) → decreases fracture risk.

3. Coaches can prioritize interventions based on these insights:
   - Adjust training load or recovery strategies.
   - Review nutrition targets.
   - Consider equipment effects (e.g., carbon shoes) if they influence risk.

   

*Key takeaway: Even though XGBoost is a complex model, SHAP allows us to explain individual predictions in actionable terms—turning data science outputs into practical athlete care decisions.*

_______________________


## Lab 2 Summary & Learning Outcomes

In this lab, we expanded our exploration of machine learning in athlete health by working with a richer dataset of 100,000 competitive female distance runners. We progressed from a simple decision tree to a more advanced XGBoost model, and learned how to make even complex models interpretable using SHAP values.

**Key takeaways:**

1. **Data Preparation:** How to encode categorical variables, handle numeric features, and split datasets for training and testing.
2. **Model Training:** How to train and visualize a simple decision tree to understand intuitive “if-then” rules linking athlete characteristics to injury risk.
3. **Advanced Modeling:** How to train an XGBoost classifier, an ensemble of decision trees, to improve predictive performance.
4. **Model Evaluation:** Understanding accuracy, precision, recall, F1 score, and ROC AUC, and how these metrics relate to practical injury prevention decisions.
5. **Explainable AI (SHAP):** How to interpret global feature importance and local predictions for individual athletes, allowing actionable insights for coaching and athlete care.
6. **Applied Insights:** Translating model outputs into real-world interventions—such as adjusting training load, recovery, nutrition, or equipment choices—to reduce overuse fracture risk.

**Learning outcomes:**

By completing this lab, students will be able to:
- Understand the difference between interpretable and black-box models in sports science contexts.
- Evaluate and compare model performance using appropriate metrics.
- Explain both global and individual predictions using SHAP values.
- Make data-driven recommendations for athlete health and performance based on machine learning outputs.